In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style = "darkgrid")

#### V1

In [ ]:
v1 = pd.read_parquet("../data/clean_insurance_claims.parquet")

# remove columns that are not useful for modelling, high cardinality and ID features
v1 = v1.drop(columns = ["policy_number", "insured_zip", "incident_location"])

# 'total_claim_amount' is the sum of the other claims
v1 = v1.drop(columns = ["total_claim_amount"])

# 'age' is highly correlated with 'months_as_customer'
v1 = v1.drop(columns = ["age"])

# 'auto_year' is a feature with no relationship to fraud and is less useful than 'auto_make'
# 'auto_model' is a high cardinality feature
v1 = v1.drop(columns = ["auto_year", "auto_model"])

# both features are similar to 'incident_date' and 'incident_state', but the incident features are more 
# directly related to the event
v1 = v1.drop(columns = ["policy_bind_date", "policy_state"])

In [ ]:
# timestamp is divided into month and day component which allows for temporal patterns to be captured 
v1["incident_date_month"] = v1["incident_date"].dt.month
v1["incident_date_day"] = v1["incident_date"].dt.day

v1 = v1.drop(columns = ["incident_date"])

In [ ]:
v1.to_parquet("../data/fraud_model_dataset_v1.parquet")

#### V2

In [ ]:
v2 = pd.read_parquet("../data/clean_insurance_claims.parquet")

v2 = v2.drop(columns = ["policy_number", "insured_zip", "incident_location", "total_claim_amount", 
                        "age", "auto_year", "auto_model", "policy_state"])

In [ ]:
# looking at the feature importance for v1 models, date columns, apart from 'incident_hour_of_the_day', seem to be of little importance
# instead of breaking down 'incident_date' into month and day, we can combine it with 'policy_bind_date' to create 'policy_duration'
# this new feature should inform us of the number of days it took for the incident to occur after the policy was set

v2["policy_duration"] = (v2["incident_date"] - v2["policy_bind_date"]).dt.days
v2 = v2.drop(columns = ["incident_date", "policy_bind_date"])

In [ ]:
palette = {"Y": "green", "N": "red"}

fig, axes = plt.subplots(1, 2, figsize = (6, 3), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Distribution of Policy Duration by Fraud", fontsize = 14, fontweight = "semibold")

sns.boxplot(data = v2, x = "fraud_reported", y = "policy_duration", ax = axes[0], hue = "fraud_reported", palette = palette)
sns.histplot(data = v2, x = "policy_duration", hue = "fraud_reported", ax = axes[1], kde = True, legend = False, palette = palette)

fig.savefig("../figures/fraud/feature_engineering/fraud_policy_duration_feature.png", dpi = 300)
plt.show()

In [ ]:
v2.to_parquet("../data/fraud_model_dataset_v2.parquet")

## V3

In [ ]:
v3 = pd.read_parquet("../data/fraud_model_dataset_v2.parquet")

In [ ]:
# the last new feature we will make is 'net_capital' which will tell us the monetary sum given to the claimant
v3["net_capital"] = v3["capital-gains"] - v3["capital-loss"]
v3 = v3.drop(columns = ["capital-gains", "capital-loss"])

In [ ]:
palette = {"Y": "green", "N": "red"}

fig, axes = plt.subplots(1, 2, figsize = (6, 3), constrained_layout = True)
axes = axes.flatten()
plt.suptitle("Distribution of Net Capital by Fraud", fontsize = 14, fontweight = "semibold")

sns.boxplot(data = v3, x = "fraud_reported", y = "net_capital", ax = axes[0], hue = "fraud_reported", palette = palette)
sns.histplot(data = v3, x = "net_capital", hue = "fraud_reported", ax = axes[1], kde = True, legend = False, palette = palette)

fig.savefig("../figures/fraud/feature_engineering/fraud_net_capital_feature.png", dpi = 300)
plt.show()

In [ ]:
v3.to_parquet("../data/fraud_model_dataset_v3.parquet")